# Quality report — districting, sensor placement, and the worlds built on them

One notebook for every study under `data/09_experiments`. It replaces `districting_report`,
`placement_report` and the comparison part of `04_districting_poc`.

**Navigation.** Pick a **study → partition → world** in the selector below. Each option says
what it is — a world reads `drift A: low industrial -> low residential | LI_LR_LI_LI ->
LR_LR_LI_LI | seed 43 | 54 mo (drift done m36) | baseline | stack 3f9c1a | ok 6875f3` — and the
card under the dropdowns spells out the selected world (seed node, horizon, coupling, operating
point, sensors, stacks, and every study it belongs to). Every section is
bound to it: sections about a world re-draw when the world changes, sections comparing across a
study (§4) re-draw when the study changes. Nothing here writes; everything is read from the
experiments store (`fedwater.reporting.Atlas`).

**What is shared, and cached once.** A *partition* (network + method) is shared by every world
cut by it; a *probe stack* by every world at the same operating point (for an every-district
study: all K drift targets × seeds of a group); a *world* is what differs (drift target, seed).
The atlas loads each shared object once — switching between worlds that share a stack is instant.

Each section ends with a **how to read** card: what the quantity measures, its definition, its
range, and the value to expect. Thresholds are printed from the selected world's own parameters.

In [1]:
%matplotlib inline
import warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import sys, pathlib

# <repo>/src must be importable: the lab borrows fedwater's own code.
REPO = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "src" / "fedwater").is_dir()), None)
assert REPO is not None, "run this from inside the fedwater repo"

# The directory that CONTAINS the `fedwater_lab/` package -- adjust to your layout.
LAB_DIR = REPO / "notebooks" / "fedwater_lab_refactor"

for p in (REPO / "src", LAB_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from fedwater import districting as dst
from fedwater.placement import browsers as B, classification as cl, figures as F
from fedwater.placement import mixture_probe as mp
from fedwater.reporting import Atlas

warnings.filterwarnings("ignore")
logging.getLogger("kedro").setLevel(logging.WARNING)
pd.set_option("display.width", 230, "display.max_columns", 60, "display.max_rows", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

EXPERIMENTS = None          # None -> <project>/data/09_experiments
ATLAS = Atlas(root=EXPERIMENTS)
SEL = ATLAS.selector()      # study -> partition -> world
display(SEL)

In [2]:
# Optional: pre-load every probe stack of the selected study (with its probe
# series) so the browsers below do not pause on first use.
# ATLAS.warm(SEL.study)

## 0. The study

Every world of the selected study, and how many built. Only `ok` worlds are offered in the
selector; the others are listed here with their status (`sim_failed`, `sim_validation_failed`,
`sim_incomplete:<artifact>`). The failure text of a world is in its manifest's `stdout_tail`.

In [2]:
def study_status(atlas, study):
    display(atlas.status(study))
    cols = [c for c in ("sim_hash", "status", "network", "districting", "drift_district",
                        "drift_change", "initial_map", "final_map", "sim_seed",
                        "n_months", "n_months_auto",
                        "drift_last_switch", "selection_stack", "label_stack", "minutes")
            if c in atlas.index]
    display(atlas.worlds(study)[cols])

SEL.show(study_status, level="study");

Output()

## 1. The network

Hydraulic components decide what a node plot hides: which district owns a source, where a pump
sits relative to a cut, which valves regulate (PRV/PSV/PBV make coupling directional). Published
DMA tags (`[TAGS]`), where a network ships them, are the only external partition AMI/ARI may be
quoted against.

In [3]:
def network_section(w):
    wn = w.network_model
    g = dst.build_graph(wn)
    print(f"{w.row['network']}: {g.number_of_nodes()} nodes, {g.number_of_edges()} edges, "
          f"components {dst.component_report(g)['sizes']}")
    print(f"{len(wn.junction_name_list)} junctions, {len(wn.tank_name_list)} tanks, "
          f"{len(wn.reservoir_name_list)} reservoirs | {len(wn.pipe_name_list)} pipes, "
          f"{len(wn.pump_name_list)} pumps, {len(wn.valve_name_list)} valves")
    print("elevation:", dst.check_elevation_informative(g))
    ref = dst.metrics.published_dma_tags(wn)
    print("published DMA tags:", f"{len(set(ref.values()))} DMAs -> AMI/ARI meaningful"
          if ref else "none -> agreement columns absent")
    display(dst.hydraulic_assets(wn).groupby(["kind", "subtype"]).size().rename("n").to_frame())

SEL.show(network_section);

Output()

## 2. Districting

### 2.1 What is on disk, and is it what the worlds used?

`status`: `current` = the stored partition matches what the current parameters and `.inp`
would build; `stale` = a dial or the network changed since; `missing` = never built.
`partition_id` is the partition's **content**, and it is part of every world's hash. The
partitions are then rebuilt in memory by the pipeline's own `build_partitions` and checked
against disk — the report refuses to draw anything that is not the stored partition.

In [4]:
def districting_store(w):
    P = ATLAS.partition(w.row["network"])
    display(P.store)
    P.results           # rebuilds + checks every stored partition against disk
    print("in-memory partitions == stored partitions:", list(P.results))
    print(f"this world: {w.row['districting']} (partition {w.row.get('partition_id')})")

SEL.show(districting_store);

Output()

### 2.2 The dials, live

Move a slider, see the partition and its metrics. The factor weights apply to `spectral`; the
community methods only take `k`. Exploring here changes nothing on disk — a setting worth
keeping goes into `parameters.yml` (`districting.spectral.weights`) and the pipeline rebuilds.

In [5]:
def dials(w):
    P = ATLAS.partition(w.row["network"])
    ref = dst.metrics.published_dma_tags(P.wn) or None
    k = len(P.manual["districts"])
    dst.plots.interactive_districting(P.wn, network=w.row["network"], reference=ref,
                                      coupling=None, k=k)

SEL.show(dials);

Output()

### 2.3 Compare two methods

Pick any two stored partitions. The second is recoloured to maximum overlap with the first, so
the eye reads the difference in boundaries, not a colour permutation; the third panel marks every
node the two disagree about. The static row below is every method's own map.

In [6]:
def compare_methods(w):
    P = ATLAS.partition(w.row["network"])
    res = P.results
    other = [m for m in res if m != w.row["districting"]]
    pair = (w.row["districting"], other[0]) if other else None
    dst.plots.interactive_comparison(P.wn, res, default=pair)
    fig, axes = plt.subplots(1, len(res), figsize=(4.2 * len(res), 4.4), squeeze=False)
    for ax, (name, r) in zip(axes[0], res.items()):
        dst.plots.plot_districts(P.wn, r.labels, ax,
                                 "%s — k=%d, imbalance %.2f"
                                 % (name, r.k, r.report["balance"]["count_imbalance"]),
                                 status=r.status if r.best else None, node_size=5,
                                 legend=(ax is axes[0][0]))
    fig.tight_layout()

SEL.show(compare_methods);

Output()

### 2.4 Sanity checks

Four blocks: the `districts.yml` **schema contract** (junctions covered exactly once, assets
recorded and never duplicated into `districts`), **partition properties** (contiguity, `k`
achieved, no empty district, every node labelled), **reproducibility** (same config → same
labels), and **hydraulics** where Phase 3 ran. The pipeline already raises on what a world cannot
survive; a failing row here carries the detail needed to decide (e.g. the rebalance tolerance can
legitimately fail when every movable boundary node is an articulation point of its district).

In [7]:
def sanity(w):
    S = ATLAS.partition(w.row["network"]).reports["sanity"]
    display(S.groupby("method")["ok"].agg(passed="sum", checks="count"))
    bad = S[~S["ok"]]
    display(bad[["method", "check", "detail"]].reset_index(drop=True) if len(bad)
            else pd.DataFrame({"note": ["every check passed for every method"]}))

SEL.show(sanity);

Output()

### 2.5 Summary, and whether each community partition is a unique answer

In [8]:
def summary(w):
    R = ATLAS.partition(w.row["network"]).reports
    display(R["summary"].set_index("method").T)
    if "ami_vs_reference" in R["summary"]:
        display(R["summary"][["method", "ami_vs_reference", "ari_vs_reference",
                              "size_imbalance", "modularity"]]
                .sort_values("ami_vs_reference", ascending=False))
    display(R["order"])

SEL.show(summary);

Output()

**How to read §2** — `n_d` junctions of district d, `k` districts, `w_e` edge weight.

| metric | measures | definition | range · expect |
|---|---|---|---|
| `size_imbalance` | how unequal the districts are | $\max_d n_d \,/\, \min_d n_d$ | ≥ 1; 1 = equal. A large value flags a tiny district: every district must hold ≥ `slots` live gauges per kind or no world can place sensors on it |
| `demand_imbalance` | the same on base demand | $\max_d Q_d / \min_d Q_d$ | ≥ 1 |
| `all_contiguous` | each district is one connected piece | induced subgraph connected ∀d | must be `True` |
| `modularity` | community structure of the cut (bare topology) | $Q=\frac{1}{2m}\sum_{ij}\big[A_{ij}-\frac{k_ik_j}{2m}\big]\delta(c_i,c_j)$ | [−½, 1]; > 0.3 is clear structure. The community methods optimise it, so they should win it |
| `structural_coupling_ratio` | how much of the graph the cut severs | $\sum_{e\,\in\,\text{cut}} w_e \,/\, \sum_e w_e$ ($w_e$ = spectral affinity, else 1) | [0, 1]; low = separable. Comparable only at the same weighting |
| `elevation_f_ratio` | do districts follow elevation bands | between- / within-district elevation variance (one-way ANOVA F) | ≥ 0; ≫ 1 in a gravity network, ≈ 1 on a flat one |
| `n_boundary_links` | cost of the cut | links whose ends lie in different districts | fewer is cheaper; meters / valves / open length need Phase 3 (`NaN` = not run, never 0) |
| `ami_vs_reference`, `ari_vs_reference` | agreement with the published DMAs | adjusted mutual information / adjusted Rand index | ≤ 1, 0 = chance. Only with `[TAGS]`; against a partition this code produced it measures self-consistency |
| `ami_between_orders`, `order_sensitive` | is the method's answer unique | AMI between runs on `.inp` vertex order vs lexical order | `order_sensitive=True`: read that row as one of several comparably modular answers |

## 3. Sensor placement — the selected world

Two stacks can sit behind one placement: the **selection** stack (reference coupling) chose
which gauges fill the slots; the **label** stack (the world's realised coupling) supplied the
`w_<District>` mixture written for them. At the reference coupling they are one stack.

### 3.1 Cache logic

A stack is identified by the hash of its spec: network, partition content, `.inp`, initial map,
operating point, probe horizon, coupling, seed, transitions and analysis thresholds — **not** the
world's drift target or seed node, nor the slots. A `failed` manifest is rebuilt on the next
request; `src_drift` flags a stack built by different code (reported, not invalidated).

In [9]:
import json
from pathlib import Path
from fedwater.placement import store

def cache_logic(w):
    ptr = w.pointer
    if ptr.get("source") != "dynamic":
        print("this world uses manual sensors: no probe stacks"); return
    display(pd.DataFrame([{"role": role, "stack_hash": ptr[role]["stack_hash"],
                           "seed": ptr[role].get("seed"), "src_drift": ptr[role].get("src_drift"),
                           "coupling": ptr[role]["coupling"]["variant"],
                           "closed": ";".join(ptr[role]["coupling"].get("closed", []))}
                          for role in ("selection", "label")]))
    print("world coupling:", ptr.get("world_coupling"), "| plan:", ptr.get("plan"))
    rows, root = [], Path(ptr["selection"]["path"]).parent
    for d in sorted(p for p in root.iterdir() if p.is_dir()):
        m = json.loads((d / "manifest.json").read_text()) if (d / "manifest.json").exists() else {}
        spec = m.get("spec", {})
        rows.append({"stack": d.name, "status": m.get("status"),
                     "coupling": (spec.get("coupling") or {}).get("variant"),
                     "months": (spec.get("plan") or {}).get("n_months"),
                     "minutes": round(float(m.get("seconds", 0)) / 60, 1),
                     "size_MB": round(sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 1e6),
                     "src_drift": m.get("src_hash") != store.src_hash(),
                     "in_use": d.name in (ptr["selection"]["stack_hash"], ptr["label"]["stack_hash"]),
                     "error": (m.get("error") or "")[:80]})
    print(f"stacks on disk for {w.row['network']}/{w.row['districting']}:")
    display(pd.DataFrame(rows))

SEL.show(cache_logic);

Output()

### 3.2 The probe design, and did the drift land where it was aimed?

One probe world per district, each converting that district's land use through
`probe.transitions` from its **initial** regime. `in_target` must be 1.00 everywhere and each
seed must sit in its own district.

In [10]:
def probe_design(w):
    S = w.selection(with_probes=True)
    display(S.seeds); display(S.horizon)
    rows = []
    for d, P in S.probes.items():
        sch, tgt = P.schedule, set(P.districts[d])
        seeded = set(sch["node"].astype(str))
        first = sch.loc[sch["drift_month"] == sch["drift_month"].min(), "node"].astype(str)
        rows.append({"drift_district": d, "n_scheduled": len(sch),
                     "in_target": round(len(seeded & tgt) / max(len(seeded), 1), 3),
                     "coverage": round(len(seeded & tgt) / len(tgt), 3),
                     "seed_node": first.iloc[0], "seed_in_target": first.iloc[0] in tgt,
                     "to_land_use": sch["to_land_use"].iloc[0],
                     "first_month": int(sch["drift_month"].min()),
                     "last_month": int(sch["drift_month"].max())})
    T = pd.DataFrame(rows); display(T)
    ok = bool((T["in_target"] == 1.0).all() and T["seed_in_target"].all())
    print("drift landing:", "OK" if ok else "FAILED — drift scheduled outside its target or seed misplaced")

SEL.show(probe_design);

Output()

### 3.3 Settling

`final` cannot start at `last_switch + ramp`: the profile is still moving there. The threshold
is self-calibrating (below). A stack whose settled window is too short is not built.

In [11]:
def settling(w):
    S, sp = w.selection(with_probes=True), w.params["sensor_placement"]["probe"]["settle"]
    display(S.settle[S.settle["drifted"]])
    fig, axes = plt.subplots(1, len(S.probes), figsize=(3.9 * len(S.probes), 3.0), squeeze=False)
    F.settle_panel(axes[0], S.probes, ref_months=sp["ref_months"], slack=sp["slack"])
    fig.tight_layout()
    print(f"ref_months={sp['ref_months']}  slack={sp['slack']}  pad={sp['pad']}")

SEL.show(settling);

Output()

**How to read §3.2–3.3**

| metric | measures | definition | range · expect |
|---|---|---|---|
| `in_target` | drift stayed in its district | $\lvert\text{scheduled}\cap\text{target}\rvert / \lvert\text{scheduled}\rvert$ | must be 1.00 |
| `coverage` | how much of the district converted | $\lvert\text{scheduled}\cap\text{target}\rvert / \lvert\text{target}\rvert$ | (0, 1]; < 1 = partial conversion (seed cannot reach all of the district from inside it) |
| `eccentricity` | the drift's minimum duration | hops from the seed to the farthest node of its district | the front advances ≤ 1 hop/month, so conversion ≥ eccentricity months |
| `r_to_terminal`, `ceiling`, `threshold` | has the profile reached its final shape | $r_m=\mathrm{corr}(\text{month }m,\ \text{terminal reference})$; ceiling = median $r$ over the `ref_months` reference months; threshold = ceiling − `slack` | settled from the first month after which $r_m \ge$ threshold **stays** true |
| `settled_months`, `ok` (horizon) | usable settled tail | months from `settle_month + pad` to the horizon | `ok=True`; fewer than needed → the stack refuses to build |

### 3.4 The quantities

**Excitation.** On the z-scored weekly profile, $\Delta z_D(k)$ is district $k$'s change of
temporal *signature* between the initial and the settled regime — a pure level change scores
zero, which is the point: level is what per-client scaling destroys, shape is what survives it.
For a gauge $s$, $\Delta z_s$ is the same difference on its own series.

In [12]:
def informative(w):
    S = w.selection()
    display(S.diagnostics); display(S.tier_sensitivity); display(S.purity_histogram)
    display(S.stability.describe().round(3))
    c = w.params["sensor_placement"]["classes"]
    print(f"thresholds: null_factor={c['null_factor']} (sweep {c['null_factors']}), "
          f"core_purity={c['core_purity']}, foreign_max={c['foreign_max']}, "
          f"mixed_min_foreign={c['mixed_min_foreign']}, min_n_mass={c['min_n_mass']}, "
          f"gate={c.get('gate', 'norm')}")

SEL.show(informative);

Output()

**How to read — the mixture.** $u_k=\Delta z_D(k)/\lVert\Delta z_D(k)\rVert$ is district $k$'s
direction of change; $\nu$ the same projection measured on split-half noise (nothing drifted).

| metric | measures | definition | range · expect |
|---|---|---|---|
| gain $g_{s,k}$ | how much of $k$'s signature reached gauge $s$ | $\langle\Delta z_s,\Delta z_D(k)\rangle/\lVert\Delta z_D(k)\rVert^2$ | regression coefficient; kept only if the cell clears the gate |
| gate (`norm`) | did the gauge respond at all | live if $\lVert\Delta z_s\rVert >$ `null_factor` $\times$ split-half noise norm | `projection` gate: $\lvert\langle\Delta z_s,u_k\rangle\rvert >$ `null_factor` $\times\,\nu_{s,k}$ |
| $w_{s,k}$ (`w_<District>`) | the gauge's blend of districts | $g^+_{s,k}/\sum_{k'} g^+_{s,k'}$ over live cells | simplex: $w\ge0$, $\sum_k w=1$ |
| `purity` | share from its own district | $w_{s,\mathrm{home}(s)}$ | [0, 1] |
| `second`, `w_second` | the largest foreign contributor | $\max_{k\ne\mathrm{home}} w_{s,k}$ | [0, 1]; the `mixed` slots need ≥ `mixed_min_foreign` and `n_mass` ≥ `min_n_mass` |
| `n_live`, `n_mass` | cells that cleared the gate / carry mass | counts over $k$ | `n_live<=1_%` high ⇒ mixtures one-hot by construction |
| `tier` | region of the purity axis | `core` ≥ `core_purity`; `foreign` ≤ `foreign_max`; `transition` between; `unusable` no live mass | a stable tier across `null_factors` means the mixture, not the threshold, decides |
| `snr_home` | response strength vs noise | $\lVert\Delta z_s\rVert$ / split-half noise | ≫ 1 is a clear response |
| `neg_mass` | anti-phase response | mass of negative gains, kept out of the simplex | report only |

### 3.5 Elasticity

$E_{s,k}$: gauge $s$'s response per unit of district $k$'s excitation, soft-thresholded against
the projection noise. Measured in `shape` (z-scored) and `native` (mca, L/s) space; where the two
disagree, a level effect is doing the talking.

In [13]:
def elasticity(w):
    S = w.selection()
    home = S.mixture.set_index("id")["home"]
    rows = []
    for space, E in S.elasticity.items():
        A = E.abs()
        kind = pd.Series(np.where(A.index.str.startswith("p_"), "pressure", "flow"), index=A.index)
        h = home.reindex(A.index)
        for k in ("flow", "pressure"):
            sub, hh = A[kind == k], h[kind == k]
            own = np.array([sub.loc[i, d] if d in sub.columns else np.nan for i, d in zip(sub.index, hh)])
            other = (sub.sum(axis=1).to_numpy() - np.nan_to_num(own)) / max(sub.shape[1] - 1, 1)
            rows.append({"space": space, "kind": k, "n": len(sub),
                         "E_own_mean": np.nanmean(own), "E_other_mean": np.nanmean(other),
                         "own_over_other": np.nanmean(own) / max(np.nanmean(other), 1e-12)})
    display(pd.DataFrame(rows).round(4))
    es = S.elasticity["shape"]; en = S.elasticity["native"].reindex_like(es)
    print("shape vs native elasticity, pooled correlation:",
          round(float(np.corrcoef(es.to_numpy().ravel(), en.to_numpy().ravel())[0, 1]), 3))

SEL.show(elasticity);

Output()

### 3.6 Dependence, response, leak, and the estimator's own checks

In [14]:
def dependence(w):
    S = w.selection()
    for kind in ("flow", "pressure"):
        if kind not in S.dependence:
            continue
        print(f"--- {kind}: R[j,k] (shape) ---")
        display(S.dependence[kind]["R"].round(3))
        display(S.dependence_summary[S.dependence_summary["kind"] == kind])
        display(S.asymmetry[kind].round(3))
        if kind in S.dependence_native:
            print(f"{kind}: R[j,k] (native)"); display(S.dependence_native[kind]["R"].round(3))
    display(S.checks)
    keys = sorted(S.response)
    fig, axes = plt.subplots(1, len(keys), figsize=(3.6 * len(keys), 3.4), squeeze=False)
    for ax, (kind, use) in zip(axes[0], keys):
        F.matrix(ax, S.response[(kind, use)], title=f"response {kind}/{use}",
                 xlabel="measured at", ylabel="drifted")
    fig.tight_layout(); plt.show()
    for kind, A in S.leak.items():
        print(f"leak ({kind}): rows drifted, columns measured"); display(A.round(3))
    display(S.carriers.head(20)); display(S.carrier_stability.head(20))

SEL.show(dependence);

Output()

**How to read §3.5–3.6** — $\mathrm{PROJ}_{s,k}=\langle\Delta z_s,u_k\rangle$,
$\mathrm{EXC}_k=\lVert\Delta z_D(k)\rVert$.

| metric | measures | definition | range · expect |
|---|---|---|---|
| $E_{s,k}$ | response per unit excitation | $\mathrm{sign}(\mathrm{PROJ})\max(\lvert\mathrm{PROJ}\rvert-\nu,0)/\mathrm{EXC}_k$ | 0 = no response; `own_over_other` ≫ 1 = gauges mostly see their own district |
| shape vs native $r$ | does the level agree with the shape | pooled correlation of the two $E$ | near 1; low = a level effect the shape hides |
| $D_{j,k}$ | how much district $j$'s meters see $k$'s drift | mean over **every** non-degenerate gauge homed in $j$ of $\lvert E_{s,k}\rvert$ | ≥ 0. Averaging over the purest gauges only would force the off-diagonal to 0 by construction |
| $R_{j,k}$ | the same, relative | $D_{j,k}/D_{j,j}$ | diagonal = 1; $R>1$: $j$'s meters respond more to $k$'s drift than to their own. Asymmetric on purpose (influence follows the supply path) |
| `in_mean` / `out_mean` / `net` | how much a district receives / emits | row / column means of $R$ off the diagonal; `net` = out − in | `strongest_source` = the $k$ with max $R_{j,k}$ |
| response `(kind, tier)` | the transpose restricted to a tier | rows drifted, columns measured | `core` should be ≈ identity (a core gauge is pure by definition) |
| `leak` | the same question via resemblance | top-3 resembling gauges per district | cross-check of $R$ |
| `checks`: `noise_pass_rate`, `diag_dominance`, `median_nu_over_proj` | the estimator's own health | share of cells above noise; diagonal vs off-diagonal; typical noise-to-signal | `median_nu_over_proj` ≪ 1 is healthy |

*Limit of the claim:* each district's demand is generated independently, so this says $j$'s
**observations** carry $k$'s demand — influence at this operating point, not causality.

### 3.7 Verification — contiguity, tiers on the map, channel diversity

A mixture is informative only if it **varies between gauges**. A `degenerate` channel reports one
blend everywhere (e.g. KY7 pressure, where the tank-and-pump state has one degree of freedom);
it is kept in the placement and flagged (`channel_verdict`).

In [15]:
def verification(w):
    S = w.selection()
    display(S.connectivity)
    TAB = cl.classification(S)
    fig, ax = plt.subplots(1, 2, figsize=(15, 6.4))
    F.tier_network(ax[0], w.topology, TAB, S.districts, "flow", title=f"{S.method} — flow tiers")
    F.tier_network(ax[1], w.topology, TAB, S.districts, "pressure", title=f"{S.method} — pressure tiers")
    fig.tight_layout(); plt.show()
    fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
    for a, k in zip(ax, ("flow", "pressure")):
        mp.plot_purity({"mixture": S.mixture, "districts": S.names}, k, ax=a)
    fig.tight_layout(); plt.show()
    CH = w.load("placement_channels"); display(CH)
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    F.channel_bars(ax, CH.rename(columns={"stack": "config_id"}), min_tv=cl.CHANNEL_MIN_TV)
    fig.tight_layout()

SEL.show(verification);

Output()

**How to read §3.7** — $W$ = the gauges × districts matrix of mixtures of one channel (kind).

| metric | measures | definition | range · expect |
|---|---|---|---|
| `components`, `largest_share` | are a district's core gauges one connected region | connected components of the core gauges on the network | one component, share ≈ 1 |
| `mean_pairwise_tv` | how different the gauges' blends are | mean over gauge pairs of $\tfrac12\sum_k\lvert w_{ik}-w_{jk}\rvert$ | [0, 1]; `ok` needs ≥ 0.25 |
| `distinct_share` | how many distinct blends | unique rows of $W$ (2 dp) / rows | [0, 1]; `ok` needs ≥ 0.12 |
| `eff_rank` | independent directions the blends span | $(\sum\sigma)^2/\sum\sigma^2$ of the centred $W$ | 1 = one degree of freedom (degenerate); up to $k-1$ |
| `verdict` | can this channel support a placement | `ok` iff both thresholds hold | `degenerate` channels are flagged, not dropped |

### 3.8 Interactive views

**Stack browser** — response/leak matrix beside a metric on the network, selection and label
stacks side by side. **Mixture browser** — tiers on the map, the per-district simplex, dependence
and response. **Lab browser** — one probe world (the one converting this world's drift district):
weekly shapes, the selection map, and the shape-distance clustering.

In [16]:
def browsers(w):
    S, L = w.selection(with_probes=True), w.label()
    B.sweep_browser({f"selection:{S.stack_hash}": S,
                     **({} if L is S else {f"label:{L.stack_hash}": L})},
                     w.topology,
                     figsize=(8, 4.0))
    B.mixture_browser(S, w.topology, figsize=(15, 5))
    d = w.row["drift_district"] if w.row["drift_district"] in S.probes else sorted(S.probes)[0]
    B.lab_browser(S.probes[d], S.scores[S.scores["drift_district"] == d])

SEL.show(browsers);

Output()

In [17]:
S = SEL.world.selection()
M, E = S.mixture.set_index("id"), S.elasticity["shape"]
f = M[M["kind"] == "flow"]
rows = [{"id": i, "home": r.home, "foreign": d, "w": r[f"w_{d}"], "absE": abs(E.loc[i, d])}
        for i, r in f.iterrows() for d in S.names if d != r.home and r[f"w_{d}"] > 0.2]
T = pd.DataFrame(rows)
own = {h: f[(f.home == h) & (f.tier == "core")].index.to_series()
           .map(lambda i: abs(E.loc[i, h])).median()
       for h in S.names}
T["absE_own_core_median"] = T["home"].map(own)
display(T.groupby(["home", "foreign"])[["w", "absE", "absE_own_core_median"]].median().round(3))
print("foreign cells with E == 0:", round((T["absE"] == 0).mean(), 2))

w   absE  absE_own_core_median
home       foreign                                       
District_A District_B  0.283  0.120                 0.791
           District_C  0.549  0.188                 0.791
           District_D  0.255  0.068                 0.791
District_B District_A  0.258  0.100                 0.971
           District_C  0.337  0.190                 0.971
           District_D  0.354  0.109                 0.971
District_C District_A  0.247  0.030                 1.000
           District_B  0.218  0.027                 1.000
           District_D  0.262  0.077                 1.000
District_D District_A  0.250  0.077                 0.956
           District_B  0.251  0.098                 0.956
           District_C  0.522  0.189                 0.956

foreign cells with E == 0: 0.0


### 3.9 The placement

One row per slot, identical slot names in every district. `filled_as`: `pure` / `mixed` by the
book, then the ladder `pure_redundant`, `mixed_redundant`, `pure_fill`, `best_available`,
`unresponsive`. `sel_*` columns are what the selection stack saw; the unprefixed mixture columns
are the label stack's.

In [18]:
def placement(w):
    PL = w.load("sensor_placement")
    W = [c for c in PL.columns if c.startswith("w_") and c != "w_second"]
    cols = ["district", "kind", "slot", "filled_as", "sensor", "role", "sel_tier", "tier",
            "purity", "second", "w_second", *W, "channel_verdict"]
    display(PL[cols].round(3))
    display(pd.crosstab([PL["kind"], PL["district"]], PL["filled_as"]))
    COV = w.load("placement_coverage")
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
    for ax, kind in zip(axes, ("flow", "pressure")):
        F.slot_fill_bars(ax, COV, kind)
    fig.tight_layout(); plt.show()
    if not w.pointer.get("same_stack"):
        moved = PL[(PL["sel_tier"] != PL["tier"]) | ((PL["sel_purity"] - PL["purity"]).abs() > 0.1)]
        print("where the label stack disagrees with the selection stack:")
        display(moved[["district", "kind", "slot", "sensor", "sel_tier", "tier",
                       "sel_purity", "purity", "sel_w_second", "w_second"]].round(3))
    else:
        print("selection and label stacks are the same stack")
    display(w.load("placement_report_arms"))

SEL.show(placement);

Output()

### 3.10 Do the selected sets make sense?

Against the **demand profile**: a gauge chosen for its mixture should still be recognisably
reading water — a flat trace against a strongly shaped demand week means the slot holds something
the district's behaviour does not reach. Against **location**: a `mixed` gauge should sit where
districts meet, a `pure` one inside its own. Then the chosen set scored on axes it was **not**
chosen on, and against this world's own drift.

In [19]:
KIND = "flow"

def sets_make_sense(w):
    S, PL = w.selection(with_probes=True), w.load("sensor_placement")
    d0 = w.row["drift_district"] if w.row["drift_district"] in S.probes else sorted(S.probes)[0]
    P = S.probes[d0]
    fig, axes = plt.subplots(len(S.names), 2, figsize=(16, 2.6 * len(S.names)), squeeze=False)
    for row, d in zip(axes, S.names):
        for ax, cls in zip(row, ("pure", "mixed")):
            ids = list(PL[(PL["district"] == d) & (PL["kind"] == KIND) & (PL["slot_class"] == cls)]["sensor"])
            F.selection_vs_demand(ax, P, ids, d, phase="final")
            ax.set_title(f"{d} — {cls} slots ({KIND}), probe world {d0}", fontsize=9)
    fig.tight_layout(); plt.show()
    ARMS = PL.assign(arm=PL["slot_class"])
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    for r, kind in enumerate(("flow", "pressure")):
        for c, cls in enumerate(("pure", "mixed")):
            F.selection_network(axes[r][c], w.topology, ARMS, S.districts, cls, kind)
    fig.tight_layout(); plt.show()
    cols = [c for c in ("sel_snr_home", "sel_served_own", "sel_response_mean",
                        "sel_r_res_own_init", "neg_mass") if c in PL.columns]
    display(PL.groupby(["kind", "slot_class", "filled_as"])[cols].mean().round(3))
    V = w.load("placement_verification")
    print("against this world's own drift:", V["status"].iloc[0]); display(V.head(40))

SEL.show(sets_make_sense);

Output()

**How to read §3.9–3.10**

| metric | measures | definition | range · expect |
|---|---|---|---|
| `filled_as` | how the slot was filled | first rung of the ladder that had an eligible gauge | by-the-book (`pure`, `mixed`) is the target; `best_available` / `unresponsive` = the district has no such gauge at this partition and operating point |
| `purity_swing`, `tier_stable` | is the tier robust to `null_factor` | max − min purity across `null_factors`; tier unchanged | swing ≤ `max_purity_swing` |
| `sel_served_own` | independent topological opinion (flow) | share of the demand fed **through** the link (its downstream subtree) that belongs to its own district | not what the selection used — agreement is evidence; disagreement usually means a loop or a pump/valve crossing |
| `sel_r_res_own_init` | shape resemblance to its own district's initial demand | correlation | uninformative on a homogeneous initial map |
| `observed_g` vs `expected_w_drift` | does this world confirm the label | the drift district's gain on each chosen gauge vs the label stack's $w$ | a report, never a gate; `unsettled` = the world's horizon is too short to verify |

In [ ]:
from fedwater.placement import signal_probe as sp
 
KIND = "flow"
 
 
def _r2(y, yhat0):
    """R² with a free intercept, per row: y, yhat0 are (n, T)."""
    e = y - yhat0
    e = e - e.mean(axis=1, keepdims=True)
    ss = ((y - y.mean(axis=1, keepdims=True)) ** 2).sum(axis=1)
    return np.where(ss > 0, 1.0 - (e ** 2).sum(axis=1) / np.where(ss > 0, ss, 1.0), np.nan)
 
 
def _windows(P, settle, patterns):
    """(init, final) step windows -- the logic of `placement.verify.verify_placement`."""
    sm, n = P.steps_month, int(P.time["n_months"])
    if float((patterns or {}).get("seasonality_scale", 0.0)) > 0:
        return mp.season_windows(P, pad=int(settle["pad"]))
    final0 = int(P.phases()["final"][0])
    try:
        s_lo, _ = mp.settled_window(P, ref_months=int(settle["ref_months"]),
                                    slack=float(settle["slack"]), pad=int(settle["pad"]),
                                    min_months=0)
        start = max(s_lo // sm, final0)
    except ValueError:
        start = final0
    need = int(settle.get("min_months", 3))
    if n - start < need:
        raise ValueError(f"{n - start} settled month(s) after the drift completes "
                         f"(month {final0}), {need} needed")
    return mp.baseline_window(P), (start * sm, n * sm)
 
 
def mixture_reconstruction(w):
    L = w.label()
    if L is None:
        print("this world uses manual sensors: no label stack"); return
    PL = w.load("sensor_placement")
    PL = PL[PL["kind"] == KIND].reset_index(drop=True)
    sch = w.load("gt_drift_schedule")
    if sch is None or not len(sch):
        print("no drift in this world: nothing to reconstruct against"); return
    tgt = str(sch.sort_values("drift_month")["district"].iloc[0])
    dy = w.load("districts")
    patterns = w.params["patterns"]
    P = sp.Probe(districts=dy["districts"], assets=dy.get("assets") or {},
                 demand=w.load("demand_series"),
                 pressures=pd.DataFrame(),           # flow only: pressures never read
                 flows=w.load("flows"), schedule=sch, time=w.params["time"],
                 params={"patterns": patterns}, drift={"tgt_district": tgt}, label="world")
 
    # a from the LABEL stack: the stack the placement's w_* came from
    G = L.gains[L.gains["kind"] == KIND]
    A = (G.assign(a=G["proj_native"] / G["excite_native"])
          .pivot(index="id", columns="drift_district", values="a"))
    assert set(A.columns) == set(P.names), (sorted(A.columns), P.names)
    A = A.reindex(index=PL["sensor"], columns=P.names)
    A_s = L.elasticity["native"].reindex(index=PL["sensor"], columns=P.names)
    W = [f"w_{d}" for d in P.names]
    lw = L.mixture.set_index("id")[W].reindex(PL["sensor"]).to_numpy()
    print(f"placement w_* vs label-stack mixture, max |diff|: "
          f"{np.nanmax(np.abs(lw - PL[W].to_numpy())):.2e}")
 
    try:
        (i0, i1), (f0, f1) = _windows(P, w.params["sensor_placement"]["probe"]["settle"], patterns)
    except ValueError as exc:
        print(f"unsettled: {exc}"); return
    cand = pd.DataFrame({"id": PL["sensor"].to_numpy(), "kind": KIND,
                         "district": PL["district"].to_numpy(), "role": "",
                         "element": PL["element"].astype(str).to_numpy()})
    Qi, sgn_i = sp.aligned_profile(P, cand, i0, i1)
    Qf, sgn_f = sp.aligned_profile(P, cand, f0, f1)
    Di, Df = sp.district_profile(P, i0, i1), sp.district_profile(P, f0, f1)
 
    a, a_s = A.to_numpy(), A_s.to_numpy()
    Hi, Hf = a @ Di, a @ Df
    out = PL[["district", "slot_class", "slot", "sensor", "tier", "purity"]].copy()
    out["R2_init"], out["R2_final"] = _r2(Qi, Hi), _r2(Qf, Hf)
    out["lam_init"] = Hi.mean(axis=1) / Qi.mean(axis=1)
    out["lam_final"] = Hf.mean(axis=1) / Qf.mean(axis=1)
    out["R2_delta"] = _r2(Qf - Qi, Hf - Hi)
    out["R2_delta_shrunk"] = _r2(Qf - Qi, a_s @ (Df - Di))
    out["sign_flip"] = sgn_i != sgn_f
    out["a_missing"] = np.isnan(a).any(axis=1)
 
    print(f"drift district: {tgt} | init steps {i0}..{i1} | final steps {f0}..{f1} "
          f"(months {f0 // P.steps_month}..{f1 // P.steps_month})")
    display(out.sort_values(["district", "slot_class", "slot"]).round(3).reset_index(drop=True))
 
 
SEL.show(mixture_reconstruction);

Output()

In [21]:
from fedwater.placement import signal_probe as sp
 
KIND = "flow"
 
 
def _stack_windows(P, settle):
    """(baseline, settled) step windows exactly as `build_mixture` takes them."""
    if float((P.params.get("patterns") or {}).get("seasonality_scale", 0.0)) > 0:
        return mp.season_windows(P, pad=int(settle["pad"]))
    return mp.baseline_window(P), mp.settled_window(
        P, ref_months=int(settle["ref_months"]), slack=float(settle["slack"]),
        pad=int(settle["pad"]))
 
 
def sign_reversal_audit(w):
    L = w.label(with_probes=True)
    if L is None or not L.probes:
        print("no label stack / no retained probe worlds"); return
    settle = w.params["sensor_placement"]["probe"]["settle"]
    M = L.mixture[L.mixture["kind"] == KIND].reset_index(drop=True)
    names = list(L.probes)
    rows = []
    for k, P in L.probes.items():
        (b0, b1), (s0, s1) = _stack_windows(P, settle)
        Q = P.flows[list(M["element"].astype(str))]
        m0 = Q.iloc[b0:b1].mean().to_numpy()
        m1 = Q.iloc[s0:s1].mean().to_numpy()
        rows.append(pd.DataFrame({"id": M["id"], "drift_district": k,
                                  "m0": m0, "m1": m1,
                                  "flip": np.sign(m0) != np.sign(m1)}))
    A = pd.concat(rows, ignore_index=True)
 
    F_ = A.pivot(index="id", columns="drift_district", values="flip").reindex(
        index=M["id"], columns=names).fillna(False).astype(bool)
    Wm = M.set_index("id")[[f"w_{d}" for d in names]].to_numpy()
    G = M[["id", "home", "tier", "purity", "second", "w_second"]].copy()
    G["n_flip"] = F_.sum(axis=1).to_numpy()
    G["flip_worlds"] = [",".join(d for d, f in zip(names, r) if f) for r in F_.to_numpy()]
    G["flip_mass"] = (Wm * F_.to_numpy()).sum(axis=1)
 
    display(Markdown("**By tier** — share of gauges that reverse in ≥1 probe world, and mean `flip_mass`"))
    display(G.groupby("tier").agg(n=("id", "size"),
                                  any_flip=("n_flip", lambda s: float((s > 0).mean())),
                                  flip_mass_mean=("flip_mass", "mean"),
                                  flip_mass_max=("flip_mass", "max")).round(3))
    display(Markdown("**By probe world** — gauges that reverse when that district drifts, per tier"))
    AT = A.merge(G[["id", "tier"]], on="id", how="left")
    AT = AT[AT["flip"]]
    display(pd.crosstab(AT["tier"], AT["drift_district"], margins=True) if len(AT)
            else pd.DataFrame({"note": ["no flow gauge reverses in any probe world"]}))
 
    PL = w.load("sensor_placement")
    PL = PL[PL["kind"] == KIND][["district", "slot_class", "slot", "sensor"]]
    display(Markdown("**The placed flow gauges**"))
    display(PL.merge(G.rename(columns={"id": "sensor"}), on="sensor", how="left")
              .sort_values(["district", "slot_class", "slot"]).round(3).reset_index(drop=True))
 
 
 
SEL.show(sign_reversal_audit);

Output()

## 4. Across the study

Everything above is one world. Here the study's partitions and stacks side by side — the
comparison the per-world reports could not make. The stack browser receives **every** stack the
study's worlds use; the districting comparison offers every partition the study built on.

In [21]:
def across_study(atlas, study):
    rows = []
    for net, meth in atlas.partitions(study):
        ws = atlas.worlds(study, net, meth, ok_only=True)
        for _, r in ws.drop_duplicates("selection_stack").iterrows():
            if not r.get("selection_path"):
                continue
            S = atlas.stack(r["selection_path"])
            ch = cl.channel_diversity(cl.classification(S)).set_index("kind")
            row = {"network": net, "districting": meth, "stack": r["selection_stack"],
                   "worlds": int((ws["selection_stack"] == r["selection_stack"]).sum())}
            for kind in ("flow", "pressure"):
                if kind in S.dependence:
                    R = S.dependence[kind]["R"].to_numpy()
                    row[f"{kind}_R_offdiag_mean"] = float(np.nanmean(R[~np.eye(len(R), dtype=bool)]))
                if kind in ch.index:
                    row[f"{kind}_verdict"] = ch.loc[kind, "verdict"]
                    row[f"{kind}_tv"] = ch.loc[kind, "mean_pairwise_tv"]
            rows.append(row)
    display(pd.DataFrame(rows).round(3))
    ver = []
    for _, r in atlas.worlds(study, ok_only=True).iterrows():
        V = atlas.world(r["sim_hash"]).load("placement_verification")
        ver.append({"sim_hash": r["sim_hash"], "districting": r["districting"],
                    "drift_district": r["drift_district"], "sim_seed": r["sim_seed"],
                    "verification": str(V["status"].iloc[0])[:90]})
    display(pd.DataFrame(ver))
    for net in sorted({n for n, _ in atlas.partitions(study)}):
        stacks = {k: v for k, v in atlas.study_stacks(study).items()}
        topo = atlas.topology(net)
        print(f"stack browser — {net}")
        B.sweep_browser(stacks, topo)
        P = atlas.partition(net)
        methods = [m for n, m in atlas.partitions(study) if n == net]
        res = {m: P.results[m] for m in methods if m in P.results}
        if len(res) >= 2:
            print(f"districting comparison — {net}")
            dst.plots.interactive_comparison(P.wn, res, default=tuple(list(res)[:2]))

SEL.show(across_study, level="study");

Output()

In [22]:
def has_native(w):
    S = w.selection()
    print("base populated:", bool(getattr(S, "base", None)))
    print("src_drift:", S.src_drift)
    print("native cols in gains:",
          [c for c in ("proj_native", "excite_native", "nu_native") if c in S.gains.columns])
    print("elasticity spaces:", list(S.elasticity))

SEL.show(has_native);

Output()

**How to read §4.** `R_offdiag_mean` is the mean of $R_{j,k}$ off the diagonal: 0 = the
partition separates the districts' signatures, larger = gauges see their neighbours' drift. Compare
it across partitions of the same network only at the same operating point. The honest sample size
for a cross-partition statement is the number of partitions, not the number of gauges.
`verification` per world: `unsettled` means the world's horizon ends before its drift settles —
use `n_months: auto`.

## 5. Read-out

**(a) Does each channel carry a placement?** §3.7. A `degenerate` channel's slots are filled but
are one reading repeated. Keep it in the world; drop it from the model with
`fl.preprocessing.channels` if the run calls for it.

**(b) Is the dependence real, or is it the threshold?** §3.4. If `n_live<=1_%` is high and the
tier counts move across `null_factors`, the mixture is one-hot by construction.

**(c) How much of the placement is by the book?** §3.9. A district with no `mixed` row has no
quantified blend to offer FL at this partition and operating point — a statement about the cut.

**(d) `R > 1`.** A ratio to the diagonal, not a fraction.

**(e) What is NOT claimed.** One operating point per stack; influence, not causality.